# Build Silver Tables

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, to_timestamp, desc, col, assert_true

timestamp_schema = "yyyy-MM-dd HH:mm:ss"

# groups together similar order_ids and puts them in order by purchase date
# .filter(col("rn") == 1) ensures that only the first purchase is kept
window = Window.partitionBy("order_id").orderBy(desc("order_purchase_timestamp"))

orders_bronze = spark.read.table("ecommerce.bronze.orders")
orders_clean = (orders_bronze
    .withColumn("order_purchase_timestamp",
        to_timestamp(col("order_purchase_timestamp"), timestamp_schema))
    .withColumn("order_approved_at",
        to_timestamp(col("order_approved_at"), timestamp_schema))
    .withColumn("order_delivered_carrier_date",
        to_timestamp(col("order_delivered_carrier_date"), timestamp_schema))
    .withColumn("order_delivered_customer_date",
        to_timestamp(col("order_delivered_customer_date"), timestamp_schema))
    .withColumn("order_estimated_delivery_date",
        to_timestamp(col("order_estimated_delivery_date"), timestamp_schema))
    .withColumn("rn", row_number().over(window))
    .filter(col("rn") == 1)
    .drop("rn")
    # Data Quality Checks: The _dq_ prefix is a naming convention — it signals to anyone reading the code that these are data quality check columns, not real data. They get dropped before the table is written so they never appear in the schema.
    .withColumn("_dq_order_id",
        assert_true(col("order_id").isNotNull(), "silver.orders: order_id cannot be null"))
    .drop("_dq_order_id")
    .withColumn("_dq_order_status",
    assert_true(
        col("order_status").isin("delivered", "shipped", "canceled", "invoiced",
                                  "processing", "approved", "unavailable", "created"),
        "silver.orders: unrecognized order_status value"
    ))
    .drop("_dq_order_status")
)

orders_clean.write.mode("overwrite").saveAsTable("ecommerce.silver.orders")
# Test comment

In [0]:
order_items_bronze = spark.read.table("ecommerce.bronze.order_items")
order_items_bronze.printSchema()

order_items_clean = (order_items_bronze
    .withColumn("price", col("price").cast("double"))
    .withColumn("freight_value", col("freight_value").cast("double"))
    .withColumn("shipping_limit_date",
        to_timestamp(col("shipping_limit_date"), timestamp_schema))
    # Data Quality Checks
    .withColumn("_dq_seller_id",
        assert_true(col("seller_id").isNotNull(), "silver.order_items: seller_id cannot be null"))
    .drop("_dq_seller_id")
    .withColumn("_dq_price",
        assert_true(col("price") >= 0, "silver.order_items: price must be non-negative"))
    .drop("_dq_price")
)

order_items_clean.printSchema()

order_items_clean.write.mode("overwrite").saveAsTable("ecommerce.silver.order_items")

In [0]:
customers_bronze = spark.read.table("ecommerce.bronze.customers")

customers_bronze.write.mode("overwrite").saveAsTable("ecommerce.silver.customers")


In [0]:
# Read ecommerce.bronze.products and join it to ecommerce.bronze.product_category_translations. 
# Use a left join on product_category_name so products without a translation still appear in the output. 
# Cast numeric columns (product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm) to IntegerType or DoubleType as appropriate.

from pyspark.sql.functions import col, cast, broadcast

products_bronze = spark.read.table("ecommerce.bronze.products")
product_category_translations = spark.read.table("ecommerce.bronze.product_category_translations")

products_join = products_bronze.join(broadcast(product_category_translations), "product_category_name", "left")

products_clean = (products_join
    .select(
        col("product_id"),
        col("product_category_name"),
        col("product_category_name_english"),
        col("product_name_lenght").cast("integer"),
        col("product_description_lenght").cast("integer"),
        col("product_photos_qty").cast("integer"),
        col("product_weight_g").cast("double"),
        col("product_length_cm").cast("double"),
        col("product_height_cm").cast("double"),
        col("product_width_cm").cast("double"),
        col("products._source_file"),
        col("products._ingested_at")
    )
    # Data Quality Checks
    .withColumn("_dq_product_weight_g",
        assert_true(
             col("product_weight_g").isNull() | (col("product_weight_g") > 0),
            "silver.products: product_weight_g must be non-negative"
        )
    )
    .drop("_dq_product_weight_g")
)

products_clean.write.mode("overwrite").saveAsTable("ecommerce.silver.products")

In [0]:
sellers_bronze = spark.read.table("ecommerce.bronze.sellers")

sellers_bronze.write.mode("overwrite").saveAsTable("ecommerce.silver.sellers")

# Test Silver Tables

In [0]:
orders_silver = spark.read.table("ecommerce.silver.orders")

# No duplicate order_ids
assert orders_silver.count() == orders_silver.select("order_id").distinct().count(), "Duplicates found"

# No null order_ids
assert orders_silver.filter(col("order_id").isNull()).count() == 0, "Null order_ids found"

# Timestamps are actual timestamps, not strings
print(orders_silver.schema["order_purchase_timestamp"].dataType)  # should be TimestampType

In [0]:
products_silver = spark.read.table("ecommerce.silver.products")

# Check that the English category name was joined in
products_silver.select("product_category_name", "product_category_name_english").show(10)

# Confirm numeric columns are no longer strings
print(products_silver.schema["product_weight_g"].dataType)  # should be IntegerType or similar

# Compare Bronze -> Silver counts

In [0]:
files = [
    "customers",
    "order_items",
    "orders",
    "products",
    "sellers",
]

for file in files:
    bronze_count = spark.read.table(f"ecommerce.bronze.{file}")
    silver_count = spark.read.table(f"ecommerce.silver.{file}")
    print(f"{file}: {bronze_count.count()} -> {silver_count.count()}")
